# Download DELVE DR3 stellar catalog cutouts for the LAPIS 2026 stream project

This notebook pulls a box of point sources from the public **DELVE DR3 Gold** catalog using
[LSDB](https://docs.lsdb.io) and saves it as a parquet file. 


In [ ]:
import os
import time

import lsdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Parameters

Pick a region. Boxes include generous padding around each stream for off-stream background.
Edit `REGION` (or add your own entry) and `OUT_DIR`.

| key | stream | approx. distance modulus | notes |
|---|---|---|---|
| `atlas` | ATLAS + Aliqa Uma | 16.8 | the practice-session region |
| `phoenix` | Phoenix | 16.2 | narrow, lower contrast |
| `tuc3` | Tucana III | 17.0 | has a progenitor; box crosses RA = 0 |

Boxes for Phoenix and Tucana III are first guesses; check them against the
[galstreams](https://github.com/cmateu/galstreams) tracks before distributing.


In [ ]:
REGIONS = {
    #  name     : (ra_min, ra_max, dec_min, dec_max)
    'atlas'   : (  5,  42, -42, -14),   # ATLAS + Aliqa Uma in the DES region
    # other interesting streams, could use galstreams to get an RA/Dec box for each
    # Phoenix
    # Tucana III
    # Jet 
}

REGION  = 'atlas'
OUT_DIR = 'data'

CATALOG      = 'https://data.lsdb.io/hats/delve/delve_dr3_gold'
CATALOG_S3   = 's3://stpubdata/mast/public/delve/hats/delve_dr3_gold'   # alternative; need to install s3fs, storage_options={'anon': True}

COLUMNS = [
    'RA', 'DEC',
    'PSF_MAG_APER_8_G_CORRECTED',   # extinction-corrected PSF mags
    'PSF_MAG_ERR_APER_8_G',
    'PSF_MAG_APER_8_R_CORRECTED',
    'PSF_MAG_ERR_APER_8_R',
    'EXT_XGB',                      # star/galaxy classifier: 0 = high-confidence star
    'FLAGS_GOLD',                   # Gold-sample quality flags: 0 = clean
]

G_MIN, G_MAX = 16, 25

# pandas .query() syntax; evaluated lazily by LSDB
ROW_FILTER = (
    'EXT_XGB == 0 '
    'and FLAGS_GOLD == 0 '
    f'and PSF_MAG_APER_8_G_CORRECTED > {G_MIN} '
    f'and PSF_MAG_APER_8_G_CORRECTED < {G_MAX}'
)
print(ROW_FILTER)

## Download

The box is split into tiles (default 5 degrees) and each tile is retried on transient network errors, so a dropped connection does not kill a 30-minute download. Set `tile_deg=360` to disable tiling.

In [ ]:
def _tiles(ra, dec, tile_deg):
    """Split an RA/Dec box into tiles no larger than tile_deg on a side.

    Handles RA ranges that wrap through 0 (e.g. ra=(350, 8)).
    """
    ra_lo, ra_hi = ra
    span = (ra_hi - ra_lo) % 360 or 360
    n_ra = max(1, int(np.ceil(span / tile_deg)))
    n_dec = max(1, int(np.ceil((dec[1] - dec[0]) / tile_deg)))
    ra_edges = [(ra_lo + span * i / n_ra) % 360 for i in range(n_ra + 1)]
    ra_edges[-1] = ra_hi % 360 if ra_hi % 360 else ra_hi
    dec_edges = np.linspace(dec[0], dec[1], n_dec + 1)
    return [((ra_edges[i], ra_edges[i + 1]), (dec_edges[j], dec_edges[j + 1]))
            for i in range(n_ra) for j in range(n_dec)]


def _download_tile(ra, dec, catalog, columns, row_filter, storage_options,
                   retries=3, wait=10):
    """Download one tile, retrying on transient network errors."""
    for attempt in range(1, retries + 1):
        try:
            cat = lsdb.open_catalog(
                catalog,
                search_filter=lsdb.BoxSearch(ra=ra, dec=dec),
                columns=columns,
                storage_options=storage_options,
            )
            return cat.query(row_filter).compute()
        except Exception as exc:                       # noqa: BLE001
            if attempt == retries:
                raise
            print(f'    attempt {attempt} failed ({type(exc).__name__}); '
                  f'retrying in {wait} s', flush=True)
            time.sleep(wait)


def download_cutout(ra, dec, catalog=CATALOG, columns=COLUMNS,
                    row_filter=ROW_FILTER, tile_deg=5.0,
                    retries=3, verbose=True):
    """Return a pandas DataFrame of DELVE stars in an RA/Dec box.

    The box is downloaded in tiles of at most ``tile_deg`` degrees on a side,
    each retried up to ``retries`` times, so that a single dropped connection
    does not kill a long download.

    Parameters
    ----------
    ra, dec : (float, float)
        Box edges in degrees. RA may wrap through 0 (e.g. (350, 8)).
    catalog : str
        HATS catalog path or URL.
    columns : list of str
        Columns to load (fewer columns = faster download).
    row_filter : str
        pandas-style query string applied lazily per partition.
    tile_deg : float
        Maximum tile size in degrees. Set large (e.g. 360) to disable tiling.
    """
    storage_options = {'anon': True} if catalog.startswith('s3://') else None
    tiles = _tiles(ra, dec, tile_deg)
    parts = []
    for k, (t_ra, t_dec) in enumerate(tiles, 1):
        t0 = time.time()
        part = _download_tile(t_ra, t_dec, catalog, columns, row_filter,
                              storage_options, retries=retries)
        parts.append(part)
        if verbose:
            print(f'  tile {k:2d}/{len(tiles)}  RA {t_ra[0]:6.1f}-{t_ra[1]:6.1f}  '
                  f'Dec {t_dec[0]:6.1f}-{t_dec[1]:6.1f}  {len(part):8,d} stars  '
                  f'{time.time() - t0:4.0f} s', flush=True)
    df = pd.concat(parts, ignore_index=True)
    return df.drop_duplicates(subset=['RA', 'DEC']).reset_index(drop=True)

In [ ]:
ra_min, ra_max, dec_min, dec_max = REGIONS[REGION]
print(f'{REGION}: RA {ra_min}-{ra_max}, Dec {dec_min}-{dec_max}')

t0 = time.time()
df = download_cutout((ra_min, ra_max), (dec_min, dec_max))
print(f'{len(df):,} stars after cuts  ({time.time() - t0:.0f} s)')
df.head()

## Sanity check

Raw stellar density on the sky. You should see survey edges, globular clusters / dwarf galaxies as
blobs, and a smooth foreground gradient. You should **not** see the stream yet.

In [ ]:
ra = df['RA'].values.copy()
if ra_min > ra_max:                       # unwrap for plotting
    ra[ra > 180] -= 360
    xlim = (ra_max, ra_min - 360)
else:
    xlim = (ra_max, ra_min)

fig, ax = plt.subplots(figsize=(11, 5))
h = ax.hexbin(ra, df['DEC'], gridsize=200, cmap='binary', mincnt=1)
ax.set_xlim(*xlim)
ax.set_xlabel('RA (deg)'); ax.set_ylabel('Dec (deg)')
ax.set_title(f'DELVE DR3 stars, {REGION}: {len(df):,} objects')
plt.colorbar(h, ax=ax, label='stars per bin')
plt.show()

print(df[['PSF_MAG_APER_8_G_CORRECTED', 'PSF_MAG_APER_8_R_CORRECTED']].describe().loc[['min', '50%', 'max']])
print('EXT_XGB values:', np.unique(df['EXT_XGB']), ' FLAGS_GOLD values:', np.unique(df['FLAGS_GOLD']))

## Save

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, f'{REGION}_cutout.parquet')
df.to_parquet(out_path)
print(f'Saved {out_path}  ({os.path.getsize(out_path) / 1e6:.0f} MB)')

## Upload

Upload the parquet file(s) to the shared Google Drive folder for the project. Suggested layout:

```
LAPIS2026_streams/
    data/
        atlas_cutout.parquet
        phoenix_cutout.parquet
        tuc3_cutout.parquet
        isochrones/
```

To do all three regions in one go, loop over `REGIONS` with the two cells under *Download* and *Save*.
The ATLAS box (~3.4 M stars, ~170 MB) takes roughly 30-40 minutes over a typical connection; the other
two are smaller.